# Hypothesis Testing: Billings Features
**Objective:** Identify statistically significant predictors of customer churn from billing-related features.
**Dataset:** `master_churn_dataset.csv`
**Target Variable:** `Prospect_Outcome` (Won vs Lost/Cancelled)

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use('ggplot')
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)

In [ ]:
# Load Data
df = pd.read_csv('../../data/processed/master_churn_dataset.csv', low_memory=False)

# Target Variable Cleanup
# Ensure target variable is binary: 1 for churned (loss), 0 for renewed (won)
# We will treat "Won" or "Renewed" as 0, and anything else like "Cancelled" or "Lost" as 1
def map_outcome(val):
    if pd.isna(val): return np.nan
    val_str = str(val).lower()
    if 'won' in val_str or 'renewed' in val_str:
        return 0
    return 1

df['Churn'] = df['Prospect_Outcome'].apply(map_outcome)

print("Dataset Shape:", df.shape)
print("Churn Distribution:")
print(df['Churn'].value_counts(dropna=False))

## 1. Feature Selection
Based on EDA, the key billing and score-related features are selected.

In [ ]:
target = 'Churn'

numeric_features = [
    'Sustainability_Score', 'Total_Renewal_Score_New', 
    'Last_Years_Price', 'Auto_Renewal_Score', 
    'Status_Scores', 'Anchoring_Score', 
    'Tenure_Scores', 'Renewal_Score_At_Release', 
    'Total_Net_Paid'
]

categorical_features = [
    'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 
    'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token', 
    'Payment_Method', 'Band'
]

## 2. Hypothesis Testing: Categorical Features
Using **Chi-Square Test of Independence**
- $H_0$: The categorical feature is independent of Churn.
- $H_1$: There is a statistically significant association with Churn.

In [ ]:
cat_results = []
print("=== Categorical Features: Chi-Square Test ===")

for feat in categorical_features:
    if feat not in df.columns:
        continue
        
    # Drop NAs
    temp_df = df[[feat, target]].dropna()
    if temp_df.empty or temp_df[feat].nunique() <= 1:
        continue
        
    contingency = pd.crosstab(temp_df[feat], temp_df[target])
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
    
    cat_results.append({
        'Feature': feat,
        'Chi2_Stat': chi2,
        'P-Value': p_val,
        'Significant (p<0.05)': p_val < 0.05
    })

cat_results_df = pd.DataFrame(cat_results)
display(cat_results_df)

# Visualizing Proportions
for feat in categorical_features:
    if feat in df.columns:
        ct = pd.crosstab(df[feat], df['Prospect_Outcome'], normalize="index")
        print(f"\n--- {feat} (Proportions) ---")
        display(ct)

## 3. Hypothesis Testing: Numeric Features
Using **Mann-Whitney U Test** (Non-parametric) and **Welch's T-Test**.
- $H_0$: There is no significant difference in the feature median/mean between Churned and Renewed customers.
- $H_1$: There is a significant difference.

In [ ]:
num_results = []
print("=== Numeric Features: Mann-Whitney U & Welch's T-Test ===")

for feat in numeric_features:
    if feat not in df.columns:
        continue
    
    temp_df = df[[feat, target]].dropna()
    
    # Group series
    churned = temp_df[temp_df[target] == 1][feat]
    renewed = temp_df[temp_df[target] == 0][feat]
    
    if len(churned) == 0 or len(renewed) == 0:
        continue
        
    # Mann-Whitney U Test
    stat_mw, p_mw = stats.mannwhitneyu(churned, renewed, alternative='two-sided')
    
    # Welch's T-Test (doesn't assume equal variance)
    stat_t, p_t = stats.ttest_ind(churned, renewed, equal_var=False)
    
    num_results.append({
        'Feature': feat,
        'MW_U_Stat': stat_mw,
        'MW_P-Value': p_mw,
        'T-Test_Stat': stat_t,
        'T_P-Value': p_t,
        'Significant MW (p<0.05)': p_mw < 0.05,
        'Significant T-Test (p<0.05)': p_t < 0.05
    })

num_results_df = pd.DataFrame(num_results)
display(num_results_df)

In [ ]:
# Visualizing Numeric Features vs Prospect_Outcome
from math import ceil

n_cols = 3
n_rows = ceil(len(numeric_features) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if i < len(axes) and col in df.columns:
        sns.boxplot(data=df, x="Prospect_Outcome", y=col, ax=axes[i], order=["Won", "Cancelled"], hue="Prospect_Outcome", legend=False)
        axes[i].set_title(f'{col} vs Churn')
        axes[i].tick_params(axis='x', rotation=45)

# Hide any unused subplots
for j in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()

## 4. Conclusion
Consolidating significant features for churn modeling.

In [ ]:
# Consolidated view of significant billing features
significant_cat = cat_results_df[cat_results_df['Significant (p<0.05)']]
significant_num = num_results_df[(num_results_df['Significant MW (p<0.05)']) | (num_results_df['Significant T-Test (p<0.05)'])]

print("Significant Categorical Features:")
display(significant_cat[['Feature', 'P-Value']])

print("\nSignificant Numeric Features:")
display(significant_num[['Feature', 'MW_P-Value', 'T_P-Value']])